# HOA Warfare — New Voice Lines (45)
Generates the 45 new lines from `data/new_voice_lines.json` (Fine From Above, perks, camo/Prestige announcer) using the same edge-tts CAST as the main studio notebook.
**Run all cells top to bottom**, then download the zip at the end.

## Step 1 — Install

In [ ]:
!pip install -q edge-tts
!ffmpeg -version 2>&1 | head -1

## Step 2 — Download the new-lines script

In [ ]:
import json, urllib.request
URL = "https://raw.githubusercontent.com/bparlette/hoa-warfare/main/data/new_voice_lines.json"
urllib.request.urlretrieve(URL, "/tmp/new_voice_lines.json")
lines = json.load(open("/tmp/new_voice_lines.json"))
print(f"{len(lines)} new lines loaded")
for l in lines[:3]:
    print(" ", l["id"], "|", l["text"][:50])

## Step 3 — Voice cast (same as studio notebook)

In [ ]:
# faction -> (edge voice id, rate %, pitch Hz)
# rate: "+25%" = 1.25x speed | pitch: "+10Hz" raises slightly
CAST = {
    # --- women ---
    "karens":         ("en-US-AvaNeural",   "+25%", "+15Hz"),
    "valleyGirls":    ("en-US-JennyNeural",    "+30%", "+10Hz"),
    "theaterKids":    ("en-GB-SoniaNeural",    "+25%",  "+0Hz"),
    "wokeLibs":       ("en-US-MichelleNeural", "+25%",  "+0Hz"),
    "announcer":      ("en-US-GuyNeural",    "+10%",  "+0Hz"),  # PA/megaphone: authoritative suburban announcer
    "mlmHuns":        ("en-US-AriaNeural",     "+30%",  "+10Hz"),
    "tradwives":      ("en-US-SaraNeural",     "+15%",  "+0Hz"),
    # --- men ---
    "fratHouse":      ("en-US-AndrewNeural",     "+25%",  "+0Hz"),
    "brainRot":       ("en-US-JasonNeural",    "+40%", "+20Hz"),
    "oldMoney":       ("en-GB-RyanNeural",     "+10%", "-10Hz"),
    "rednecks":       ("en-US-DavisNeural",    "+25%",  "+0Hz"),
    "gymRats":        ("en-US-GuyNeural",      "+25%",  "+0Hz"),
    "cloutStreamers": ("en-US-BrandonNeural",   "+35%",  "+5Hz"),
    "wallstreetBets":         ("en-US-ChristopherNeural","+25%",  "+0Hz"),
    "hoaDads":        ("en-US-RogerNeural",    "+25%",  "+0Hz"),
    "emoKids":        ("en-US-BrianNeural",    "+10%", "-15Hz"),
    "fabulous":        ("en-AU-WilliamNeural",  "+25%",  "+5Hz"),
    "trumpers":       ("en-US-SteffanNeural",  "+25%",  "+0Hz"),
    "furries":        ("en-GB-ThomasNeural",   "+30%", "+10Hz"),
    "animeWeebs":     ("en-US-EricNeural",     "+35%", "+15Hz"),
    "pickleballRetirees": ("en-AU-NeilNeural",  "+15%", "-10Hz"),
}
print(f"{len(CAST)} voices")
missing = [l["faction"] for l in lines if l["faction"] not in CAST]
assert not missing, f"missing voices: {set(missing)}"
print("all factions covered")

## Step 4 — Generate 45 .ogg files

In [ ]:
import asyncio, edge_tts, subprocess, hashlib
from pathlib import Path

OUT = Path("/tmp/vo_new")
OUT.mkdir(exist_ok=True)

def valid_ogg(p):
    if not p.exists() or p.stat().st_size < 500:
        return False
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries",
                        "format=duration", "-of", "csv=p=0", str(p)],
                       capture_output=True, text=True)
    try:
        return float(r.stdout.strip()) > 0.2
    except ValueError:
        return False

async def synth_one(l):
    lid, fid, text = l["id"], l["faction"], l["text"]
    ogg = OUT / (lid.replace(".", "_") + ".ogg")
    if valid_ogg(ogg):
        return ("skip", lid)
    voice, rate, pitch = CAST[fid]
    mp3 = OUT / (lid.replace(".", "_") + ".mp3")
    try:
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        await comm.save(str(mp3))
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(mp3),
                        "-c:a", "libvorbis", "-q:a", "4", str(ogg)], check=True)
        mp3.unlink()
        return ("ok", lid)
    except Exception as e:
        return ("fail", f"{lid}: {e}")

async def main():
    done, failed, skipped = 0, [], 0
    for i in range(0, len(lines), 8):
        chunk = lines[i:i+8]
        for status, info in await asyncio.gather(*[synth_one(l) for l in chunk]):
            if status == "ok": done += 1
            elif status == "skip": skipped += 1
            else: failed.append(info)
        print(f"  {i+len(chunk)}/{len(lines)} done={done} skipped={skipped} failed={len(failed)}", flush=True)
    return done, skipped, failed

done, skipped, failed = await main()
print(f"done={done} skipped={skipped} failed={len(failed)}")
for f in failed[:10]:
    print("  FAIL:", f)
assert not failed, "synthesis failures"

## Step 5 — Measure + build manifest entries, zip for download

In [ ]:
import subprocess, hashlib, datetime

def probe_duration(p):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries",
                        "format=duration", "-of", "csv=p=0", str(p)],
                       capture_output=True, text=True)
    return round(float(r.stdout.strip()), 6)

def probe_peak(p):
    r = subprocess.run(["ffmpeg", "-i", str(p), "-af", "volumedetect",
                        "-f", "null", "/dev/null"],
                       capture_output=True, text=True, stderr=subprocess.STDOUT)
    for line in r.stdout.splitlines():
        if "max_volume" in line:
            v = line.split("max_volume:")[1].strip().split()[0]
            return None if v == "n/a" else float(v)
    return None

entries = {}
for l in lines:
    lid = l["id"]
    ogg = OUT / (lid.replace(".", "_") + ".ogg")
    h = hashlib.sha256()
    with open(ogg, "rb") as f:
        h.update(f.read())
    entries[lid] = {
        "file": ogg.name,
        "faction": l["faction"],
        "event": l["event"],
        "text": l["text"],
        "voice": CAST[l["faction"]][0],
        "duration": probe_duration(ogg),
        "peak_db": probe_peak(ogg),
        "sha256": h.hexdigest(),
    }
    d = entries[lid]["duration"]
    assert d <= 1.9, f"{lid} too long: {d}s"

manifest_add = {
    "generated": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "count": len(entries),
    "lines": entries,
}
json.dump(manifest_add, open(OUT / "manifest_add.json", "w"), indent=1)
print(f"{len(entries)} entries, all <= 1.9s")
worst = max(entries.items(), key=lambda kv: kv[1]["duration"])
print("longest:", worst[0], worst[1]["duration"], "s")

In [ ]:
!cd /tmp/vo_new && rm -f *.mp3 && zip -q -r /tmp/hoa_new_voices.zip *.ogg manifest_add.json && ls -lh /tmp/hoa_new_voices.zip
from google.colab import files
files.download("/tmp/hoa_new_voices.zip")
print("\nDone — hand the zip to Moth")